# Forty lines of nested constructors, and one wrong argument

A two-treatment surface with carryover, a hierarchical intercept, an interaction and
seasonality is a real object with real structure, and building it by hand is four nested
constructor calls deep. It works. It is also unreadable in review, and the failure mode is a
keyword landing on the wrong object — a `numeraire` on a treatment, a scale meant for the
intercept applied to the interaction — which constructs fine and changes what the model means.

`axiom.build` is the fluent layer over every `Spec`. A builder is a frozen dataclass holding an
immutable `Fields` mapping; every method returns a *new* builder, `build()` returns a `Spec`
that round-trips through `load_spec`, and a missing or misapplied input is a `BuildError` that
names what to set. `Builder` is the protocol they all satisfy — there is no inheritance
hierarchy.

In [ ]:
from axiom.build import (
    Builder, BuildError, EntitySpec, Fields, GraphBuilder, MetaBuilder, MomentFamily, PriorBuilder,
    SchedulePattern, StudyBuilder, SurfaceBuilder, VariableBuilder, VariableKind,
)
from axiom.calibrate import amplitude_prior
from axiom.core import D, Outcome, Prior, TimeWindow, Treatment, load_spec
from axiom.surface import HillKernel

from axiom.display import enable
from axiom.viz import causal_graph

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, lines

enable();  # every axiom result renders itself from here on

## `Fields`, `Builder`, `BuildError`

`Fields` is the copy-on-write store every builder wraps: `with_` adds or removes (`None`)
keys and returns a new `Fields`; `require` raises `BuildError` naming the missing keys.

In [ ]:
f = Fields({"a": 1})
g = f.with_(b=2)
print("original untouched:", "b" not in f, "| new:", g.to_dict(), "| removed:", g.with_(a=None).to_dict())
try:
    f.require("z", builder="demo")
except BuildError as e:
    print("BuildError:", e)
print("builders satisfy the protocol:", isinstance(SurfaceBuilder(), Builder), isinstance(PriorBuilder(), Builder))

## `PriorBuilder`

Family + hyper-parameters → `core.Prior`; hyper-parameters may name parent parameters for a
hierarchy. `from_moments(mean, sd, family)` matches `calibrate.amplitude_prior` for the three
`MomentFamily` values.

In [ ]:
p = PriorBuilder().normal(mu=0.0, sigma=2.0).build()
print(p, "| round-trips:", load_spec(p.to_json()) == p)
print("hierarchical parents:", PriorBuilder().normal(mu="alpha_mean", sigma="alpha_sd").build().parents)
print(PriorBuilder().family("gamma").hyper(alpha=2.0, beta=1.0).build())
print(PriorBuilder().fixed(3.0).build())
family: MomentFamily = "lognormal"
print("from moments:", PriorBuilder.from_moments(2.0, 0.5, family).build() == amplitude_prior(2.0, 0.5))
try:
    PriorBuilder().family("normal").build()
except BuildError as e:
    print("incomplete:", e)

## `VariableBuilder`

One of the five `VariableKind`s fixes the entity class; `dimension`, `measured_in`, and
`describe` apply to all, `numeraire` to a dose, `aggregation` to an outcome, and
`cluster`/`individual`/`aggregate` to a unit. The result is an `EntitySpec`.

In [ ]:
kind: VariableKind = "treatment"
t: EntitySpec = VariableBuilder().kind(kind).name("x").dimension(D.currency).measured_in("USD").build()
print(t)
print(VariableBuilder().outcome("y").dimension(D.outcome).aggregation("mean").build())
print(VariableBuilder().dose("x").dimension(D.currency).numeraire("USD").build())
print(VariableBuilder().unit("region").dimension(D.entity).cluster().build())
print(VariableBuilder().covariate("w").dimension(D.entity).describe("weight").build())
try:
    VariableBuilder().treatment("x").dimension(D.currency).numeraire("USD").build()
except BuildError as e:
    print("misapplied field:", e)

## `SurfaceBuilder`

Treatments (by name or entity) with their kernel and carryover families and parameters, the
outcome, intercept kind, interactions, nuisance terms, and likelihood → `SurfaceSpec`.

In [ ]:
spec = (
    SurfaceBuilder()
    .name("demo")
    .treatment("a", kernel="hill", reference_dose=50.0, carryover="geometric", max_lag=3)
    .treatment("b", unit="USD", kernel="logistic", amplitude_scale=2.0)
    .outcome("y", unit="units")
    .intercept("hierarchical", units=("u0", "u1"), scale=0.5)
    .interaction("a", "b", scale=0.3)
    .nuisance(fourier=(12.0, 2), trend=True)
    .likelihood("student_t", noise_scale=0.7, df=5.0)
    .build()
)
print(spec.treatment_names, spec.kernels, spec.carryover, spec.nuisance.terms, spec.likelihood)
print("round-trips:", load_spec(spec.to_json()) == spec)
print("default kernel:", SurfaceBuilder().treatment("a", reference_dose=5.0).outcome("y").build().kernels == {"a": HillKernel(reference_dose=5.0)})
base = SurfaceBuilder().treatment("a")
try:
    base.treatment("a")
except BuildError as e:
    print("refused:", e)

In [ ]:
import numpy as np

from axiom.core import Data, value

grid = {"a": np.linspace(0.0, 150.0, 120)}
dose_node = Data(name="a", dimension=D.currency)
curves = {}
for label, amplitude in (("beta_a = 5", 5.0), ("beta_a = 10", 10.0)):
    kernel = spec.kernel_of("a")
    curves[label] = np.ravel(value(kernel.response(dose_node, "a"), data=grid,
                                   params={"k_a": 50.0, "s_a": 2.0, "beta_a": amplitude}))
fig = lines(
    grid["a"], curves,
    title="What those nine lines actually built",
    subtitle="the response kernel the builder attached to treatment 'a', at two amplitudes",
    x_title="dose of a (USD)", y_title="response",
)
caption(fig, "The builder is not a wrapper around a config dict — every call lands on a typed "
             "field of a Spec, and this curve is that Spec being evaluated. A misapplied "
             "keyword is a BuildError before it can become a differently-shaped curve.")

## `StudyBuilder`

One study description builds several design objects: `build()` → `SimulationSpec`,
`build_candidate()` → `DesignCandidate`, `build_schedule()` → `Schedule` (one of the
`SchedulePattern`s), and `measured(...)`.`build_measurement()` → `calibrate.Measurement`.

In [ ]:
pattern: SchedulePattern = "pulse"
study = (
    StudyBuilder()
    .name("holdout_a")
    .method("difference_in_differences")
    .size(n_units=12, n_periods=8, n_pre=4, n_treated=6)
    .variance(unit_sd=1.0, noise_sd=0.5, period_sd=0.2, rho=0.3)
    .effect(0.4)
    .simulations(50, seed=3)
    .mass(0.9)
    .holdout(0.5)
    .precision(0.2)
    .cost(100.0, cooldown_periods=2)
    .schedule(pattern, high=1.0, low=0.0, on=2, off=2, treatment="x")
)
sim = study.build()
print(type(sim).__name__, sim.n_units, sim.n_periods, sim.effect)
cand = study.build_candidate()
print(type(cand).__name__, cand.name, cand.method, cand.holdout_fraction, cand.cost)
sched = study.build_schedule()
print(type(sched).__name__, sched.doses if hasattr(sched, "doses") else sched)

In [ ]:
patterns = {}
for name, params in (("pulse", {"high": 1.0, "low": 0.0, "on": 2, "off": 2}),
                     ("alternating", {"high": 1.0, "low": 0.0}),
                     ("constant", {"dose": 0.5})):
    built = study.schedule(name, treatment="x", **params).build_schedule()
    patterns[name] = built.doses
fig = lines(
    np.arange(len(patterns["pulse"])), patterns,
    title="One study description, three schedules",
    subtitle="the dose path each SchedulePattern produces over the study's eight periods",
    x_title="period", y_title="dose",
)
caption(fig, "The same builder that produces the SimulationSpec and the DesignCandidate "
             "produces the schedule, so the periods, the treatment name and the sizes cannot "
             "drift between the three objects a design conversation actually uses.")
estimand = GraphBuilder().edge("x", "y").estimand(Treatment(name="x", dimension=D.currency, unit="USD"), Outcome(name="y", dimension=D.outcome), dose=10.0, window=TimeWindow(start=0, stop=4))
m = study.measured(estimand, 0.3, 0.1, source="study:holdout_a").build_measurement()
print(type(m).__name__, m.estimate, m.se, m.method, m.source, "| round-trips:", load_spec(m.to_json()) == m)
try:
    StudyBuilder().size(n_periods=4).schedule("alternating").build_schedule()
except BuildError as e:
    print("BuildError:", e)

## `MetaBuilder`

Study records by hand or from a frame → `meta.Corpus`; the same builder also yields the
`PoolSpec` for pooling that corpus.

In [ ]:
mb = (
    MetaBuilder()
    .name("evidence")
    .family("fertilizer")
    .study("s1", estimate=0.4, se=0.1, read="experiment", contributor="lab_a")
    .study("s2", estimate=0.5, se=0.2, read="model", contributor="lab_b", n=30)
    .moderators("season")
    .bias_term()
    .priors(mu_scale=2.0, tau_fixed=0.1)
    .interval(mass=0.9, definition="hdi")
)
corpus = mb.build()
print(corpus.name, [r.study for r in corpus.records], "| round-trips:", load_spec(corpus.to_json()) == corpus)
print(mb.build_pool_spec())
print("from a frame:", [r.study for r in MetaBuilder().from_frame(corpus.to_frame()).build().records])

## `GraphBuilder`

Directed and bidirected edges, unmeasured nodes, isolated nodes → `identify.CausalGraph`;
`estimand(treatment, outcome, ...)` builds an `Estimand` on two graph nodes.

In [ ]:
gb = GraphBuilder().name("toy").edge("Z", "X").edge("X", "Y").bidirected("X", "W").edge("W", "Y").unmeasured("W").node("I")
g = gb.build()
print(g.name, sorted(g.nodes), "| round-trips:", load_spec(g.to_json()) == g)
print(GraphBuilder().edges("Z -> X, X -> Y").build() == GraphBuilder().edge("Z", "X").edge("X", "Y").build())
causal_graph(g, height=320)

In [ ]:
e = gb.estimand(Treatment(name="X", dimension=D.currency, unit="USD"), Outcome(name="Y", dimension=D.outcome), kind="marginal", dose=10.0, name="m")
print(e.name, e.quantity.kind, e.dimension)
try:
    gb.estimand(Treatment(name="Q", dimension=D.currency), Outcome(name="Y", dimension=D.outcome), dose=1.0)
except BuildError as err:
    print("BuildError:", err)

## The one that files things

Every builder above turns arguments into a `Spec`. This one is different in kind: it takes the
specs the other layers produce and puts them somewhere, under one experiment id, in one
party's scope.

That is not a small job, because it is the one nothing did. `design.assign` returns an
`ArmAssignment`, `design.collision` an `Occupancy`, `design.stopped_estimate` a corrected pair,
`io.ExperimentRun` has a `roles` mapping waiting for all of them, and every note that built one
of those ended with some version of *"nothing files it yet"*.

`ExperimentBuilder` adds no statistics. What it adds is that the plan, the assignment, the
readout and the correction end up together, with the run's conformance verdict standing over
them.

In [ ]:
import tempfile
from pathlib import Path

from axiom.build import ExperimentBuilder, readouts_across
from axiom.calibrate import Measurement
from axiom.design import (
    ArmAllocation, LookSchedule, Occupancy, StoppingRule, assign, monitor, pocock,
)
from axiom.estimands import Estimand, Level, Quantity
from axiom.io import Catalog, Program
from axiom.display import table
from axiom.core import (
    D, Intervention, Outcome, Population, TimeWindow, Treatment,
)

lift = Estimand(
    name="lift", quantity=Quantity(kind="contrast"),
    treatment=Treatment(name="course", dimension=D.currency, unit="USD"),
    intervention=Intervention(doses={"course": 1.0}),
    reference=Intervention(doses={"course": 0.0}),
    outcome=Outcome(name="score", dimension=D.outcome, unit="pt", aggregation="mean"),
    population=Population(name="enrolled"), window=TimeWindow(start=0, stop=8),
    level=Level(unit="individual"), dimension=D.outcome,
)

catalog = Catalog(Path(tempfile.mkdtemp()) / "book")
northwind = Program(party="northwind", program="growth")
catalog.register(northwind)
builder = ExperimentBuilder(catalog, northwind)

roster = tuple(f"u{i:04d}" for i in range(500))
arms = assign(roster, ArmAllocation.equal("control", "treated"), salt="NW-14")
weeks = Occupancy(experiment="NW-14", units=("london", "leeds"),
                  window=TimeWindow(start=0, stop=8), treatments=("course",))

run = builder.plan("NW-14", estimand=lift, roles={"assignment": arms.spec, "occupancy": weeks})
run = builder.read(builder.start(builder.commit(run)),
                   Measurement(estimand=lift, estimate=2.4, se=0.5, source="NW-14"))
print(run.summary())

### The one place it does more than plumb

A study that crossed a boundary has a biased naive estimate, and `design.stopped_estimate`
corrects it. Filing the *corrected* number and a `"stopped_early"` deviation naming the naive
one are two moves that belong together — doing them separately is how they come apart, and how
a corrected estimate ends up in a corpus with a plan that says it ran to completion.

`read_stopped` does both, and the run's conformance drops to `downgraded` on its own.

In [ ]:
looks = LookSchedule(labels=("L1", "L2", "L3", "L4"), information=(0.25, 0.5, 0.75, 1.0))
rule = StoppingRule(name="Pocock-4", looks=looks, boundaries=(pocock(0.05, looks),))
path = monitor(rule, [0.5, 0.9, 2.6], effects=[1.0, 1.8, 4.2], ses=[2.0, 1.4, 1.1])

stopped = builder.plan("NW-15", estimand=lift, roles={"assignment": arms.spec})
stopped = builder.start(builder.commit(stopped))
stopped, correction = builder.read_stopped(stopped, path, lift)

table([["naive", f"{correction.naive_drift:.4f}", f"{correction.naive_effect:.4f}"],
       ["median-unbiased", f"{correction.drift:.4f}", f"{correction.effect:.4f}"]],
      headers=("estimate", "drift", "effect (pt)"),
      title=f"{correction.rule} stopped at look {correction.look + 1}")
print(stopped.summary())
print("\ndeviation:", stopped.deviations[0].role, "|", stopped.deviations[0].reason)

### And what a programme reads

`readouts_across` collects the filed runs of several parties into the input
`design.program` and `design.online` take — and it **skips a run whose own conformance is
blocked**, because a readout the run calls blocked is not evidence about anything and quietly
including it in a book's error control is the failure `io.ExperimentRun` exists to prevent.

In [ ]:
builder.define_outcome(lift, note="the outcome the NW-14 analysis used")
table([[r.experiment, r.stage, r.conformance().status] for r in builder.runs()],
      headers=("experiment", "stage", "conformance"), title="everything filed in this scope")
print("readouts a programme would take:", [r.key for r in readouts_across([builder])])
print("terms this party has defined:", builder.definitions.names(northwind))
print(repr(builder))

## What this bought you

Every spec in the package constructible in a form a reviewer can read down the page, with each
call validated against the field it lands on, an immutable builder that can be branched and
reused, and a `BuildError` that names the missing input instead of a `TypeError` four frames
into pydantic.